In [2]:
# Train and fine-tune a Decision Tree for the moons dataset by following these steps:
# a. Use make_moons(n_samples=10000, noise=0.4) to generate a moons
#dataset.
# b. Use train_test_split() to split the dataset into a training set and a test set.
# c. Use grid search with cross-validation (with the help of the GridSearchCV
#class) to find good hyperparameter values for a DecisionTreeClassifier.
# Hint: try various values for max_leaf_nodes.
# d. Train it on the full training set using these hyperparameters, and measure
#your model’s performance on the test set. You should get roughly 85% to
#87% accuracy.


    

SyntaxError: invalid syntax (3811647073.py, line 3)

In [46]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

# 1. Get the data
moons = make_moons()
X, y = make_moons(n_samples=10000, noise=0.4)

# 2. Split the model for cross validation
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # reproducibility
)

# 3. Define model
tree = DecisionTreeClassifier(random_state=42)

# 4. Define hyperparameter grid
param_grid = {
    #"max_depth": [1, 2, 3, 4, 5, 10, 15, None],
    "min_samples_split": [2, 3, 5, 10],
    #"min_samples_leaf": [2, 3, 5, 10],
    "max_leaf_nodes": list(range(2, 10)),
#    "max_features": [None, "sqrt", "log2"],
   "criterion": ["gini", "entropy", "log_loss"]
}

# 5. Grid search with cross-validation
grid_search_cv = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="accuracy",
    n_jobs=-1            # use all cores
)

# 6. Fit
grid_search_cv.fit(X_train, y_train)

# 7. Best results
print("Best parameters:", grid_search_cv.best_params_)
print("Best CV score:", grid_search_cv.best_score_)


# Evaluate test accuracy
print(grid_search_cv)
best_model = grid_search_cv.best_estimator_

test_accuracy = best_model.score(X_test, y_test)
print("Test accuracy:", test_accuracy)

Best parameters: {'criterion': 'gini', 'max_leaf_nodes': 4, 'min_samples_split': 2}
Best CV score: 0.850375
GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_leaf_nodes': [2, 3, 4, 5, 6, 7, 8, 9],
                         'min_samples_split': [2, 3, 5, 10]},
             scoring='accuracy')
Test accuracy: 0.842


In [47]:
print(grid_search_cv)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_leaf_nodes': [2, 3, 4, 5, 6, 7, 8, 9],
                         'min_samples_split': [2, 3, 5, 10]},
             scoring='accuracy')


In [58]:
# 8. Grow a forest by following these steps:

# a. Continuing the previous exercise, generate 1,000 subsets of the training set,
# each containing 100 instances selected randomly. Hint: you can use Scikit-
# Learn’s ShuffleSplit class for this.

from sklearn.model_selection import ShuffleSplit
from sklearn.metrics import accuracy_score
import numpy as np

n_trees = 1000
n_instances = 100

mini_sets = []

rs = ShuffleSplit(n_splits=n_trees, test_size=len(X_train) - n_instances, random_state=42)
print(rs)
for mini_train_index, mini_test_index in rs.split(X_train):
     X_mini_train = X_train[mini_train_index]
     y_mini_train = y_train[mini_train_index]
     mini_sets.append((X_mini_train, y_mini_train))



# b. Train one Decision Tree on each subset, using the best hyperparameter
# values found in the previous exercise. Evaluate these 1,000 Decision Trees
# on the test set. Since they were trained on smaller sets, these Decision
# Trees will likely perform worse than the first Decision Tree, achieving only
# about 80% accuracy.

from sklearn.base import clone

forest = [clone(grid_search_cv.best_estimator_) for _ in range(n_trees)]
#print(forest)

accuracy_scores = []

for tree, (X_mini_train, y_mini_train) in zip(forest, mini_sets):
    tree.fit(X_mini_train, y_mini_train)
    
    y_pred = tree.predict(X_test)
    accuracy_scores.append(accuracy_score(y_test, y_pred))

np.mean(accuracy_scores)

# c. Now comes the magic. For each test set instance, generate the predictions
# of the 1,000 Decision Trees, and keep only the most frequent prediction
# (you can use SciPy’s mode() function for this). This approach gives you
# majority-vote predictions over the test set.

Y_pred = np.empty([n_trees, len(X_test)], dtype=np.uint8)
print(Y_pred)

for tree_index, tree in enumerate(forest):
    #print(tree_index)
    #print(tree.predict(X_test))
    Y_pred[tree_index] = tree.predict(X_test)

from scipy.stats import mode

y_pred_majority_votes, n_votes = mode(Y_pred, axis=0)
print(y_pred_majority_votes)

# d. Evaluate these predictions on the test set: you should obtain a slightly
# higher accuracy than your first model (about 0.5 to 1.5% higher).
# Congratulations, you have trained a Random Forest classifier!

accuracy_score(y_test, y_pred_majority_votes.reshape([-1]))


ShuffleSplit(n_splits=1000, random_state=42, test_size=7900, train_size=None)
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
[0 1 0 ... 1 0 0]


0.8465